In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PySpark_Venv_Setup") \
    .master("local[*]") \
    .getOrCreate()
from pyspark.sql.functions import col, sum, count, when
print("Spark Session Created Successfully")
print(spark)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/14 12:04:19 WARN Utils: Your hostname, krishna-Nitro-AN515-52, resolves to a loopback address: 127.0.1.1; using 10.72.97.241 instead (on interface wlp0s20f3)
26/05/14 12:04:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/14 12:04:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Session Created Successfully


In [5]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("data.csv")

df.show(5)

+------+------+-------+-----------+
|region|amount| status|   category|
+------+------+-------+-----------+
| North|   100|SUCCESS|Electronics|
| North|   200| FAILED|Electronics|
| South|   300|SUCCESS|   Clothing|
| South|   400|SUCCESS|Electronics|
|  East|   150|SUCCESS|    Grocery|
+------+------+-------+-----------+
only showing top 5 rows


In [ ]:
result = df.filter(col("status") == "SUCCESS") \
  .groupBy("region") \
  .agg(
      sum("amount").alias("total_sales"),
      count("amount").alias("txn_count")
  )

result.show()

In [14]:
result = df.groupBy("region").agg(
    sum(when(col("status") == "SUCCESS", col("amount")).otherwise(0)).alias("total_success_sales"),
    sum(when(col("status") == "FAILED", 1).otherwise(0)).alias("failed_txn_count")
)

result.show()

+------+-------------------+----------------+
|region|total_success_sales|failed_txn_count|
+------+-------------------+----------------+
| South|                700|               0|
|  East|                150|               1|
|  West|               1200|               0|
| North|                100|               1|
+------+-------------------+----------------+



In [7]:
df.count()

8